In [1]:
import json
import os 

with open('config.json', 'r') as f:
    data = json.load(f)

pathway_gen = os.path.abspath(data["python_files"])
pathway_temp = os.path.abspath(data["publications"])
pathway = os.path.join(pathway_temp, "manrique2015age")
original_data_pathway = os.path.join(pathway, "original_data")

# complete_path_1 = os.path.join(original_data_pathway, "einstellung.sav")
complete_path_1 = os.path.join(original_data_pathway, "requested_data.csv")

out_pathway = os.path.join(pathway, "standardized_data")
if not os.path.exists(out_pathway):
    os.makedirs(out_pathway)

In [2]:
import pandas as pd
import numpy as np
import pyreadstat

df = pd.read_csv(complete_path_1)
# original_data_pathway_out = os.path.join(original_data_pathway, 'einstellung.csv')
# df.to_csv(original_data_pathway_out, encoding='utf-8-sig', index=False)


In [3]:
df['study_id']="manrique2015age"
df.columns = map(str.lower, df.columns)
df=df.applymap(lambda s: s.lower() if type(s) == str else s)


In [4]:
df.rename(columns={"name": "ape",
    "species":"species_original"}, inplace=True)


In [5]:
comp_path_name_errors = os.path.join(pathway_gen, "common_name_errors.csv")

df_name  = pd.read_csv(comp_path_name_errors)
df['ape'] = df['ape'].str.rstrip()
for x,y in zip(df_name['wrong'],df_name['right']):
    df['ape'].replace(x, y, inplace=True)

comp_path_ape_info = os.path.join(pathway_gen, "apes_includeindatabase.csv")
apedf = pd.read_csv(comp_path_ape_info)    
df= df.merge(apedf,left_on='ape', right_on='name', how='left')

In [6]:
# df['date'] = df['date'].astype(str).str.pad(8, 'left', '0')
# df['year'] = df['date'].str.slice(4,8)
# df['day'] = df['date'].str.slice(0,2)
# df['month'] = df['date'].str.slice(2,4)
# df['year'] = '20' + df['year'].astype(str)


In [7]:
df['ape'].replace('', np.nan, inplace=True)
df.dropna(subset=['ape'], inplace=True)

df.rename(columns={"ape": "participant",
                   'age':'age_in_years'}, inplace=True)
df.columns = df.columns.str.replace(' ', '_', regex=True)
df.columns = df.columns.str.replace('\#', 'number', regex=True)
# df.columns

In [8]:
manrique2015age_standardized=df[['study_id','participant', 'age_in_years','sex','species', 
       'number_training_trials', 'number_errors_post-reversal_before_the_first_solution',
       'number_errors_post-reversal_after_the_first_solution']]
comp_out_path_stand = os.path.join(out_pathway, 'manrique2015age_standardized.csv')
manrique2015age_standardized.to_csv(comp_out_path_stand, encoding='utf-8-sig', index=False)


names =manrique2015age_standardized.columns.tolist()
df = pd.DataFrame(names)
df = df.rename(columns={0: "column_name"})
df["description"] = ""
manrique2015age_glossary=df[["column_name", "description"]]

comp_out_path_glossary = os.path.join(out_pathway, 'manrique2015age_glossary.csv')
manrique2015age_glossary.to_csv(comp_out_path_glossary, encoding='utf-8-sig', index=False)
